# Checkpoint — Recommender systems

Dit notebook wordt automatisch gecheckt via pytest (GitHub Actions). De checks voeren je notebook uit en valideren of bepaalde variabelen bestaan en binnen de verwachte ranges vallen. **Je krijgt geen hints in de foutmeldingen: het is aan jou om correct werkende code te schrijven.**

## BELANGRIJK: gebruik exact de genoemde variabelennamen

De checks zoeken volgende variabelen in je notebook. Andere namen worden niet gezien:

- `user_item_matrix` — de user-item-matrix uit de speelgoed-data (pandas DataFrame, rijen = users, kolommen = songs, ontbrekende ratings = 0)
- `sparse_user_item_matrix` — dezelfde matrix in sparse **CSR**-formaat (`scipy.sparse.csr_matrix`)
- `als_toy_model` — een **getraind** `AlternatingLeastSquares`-model op de speelgoed-matrix
- `antwoord_feedback` — meerkeuzevraag 1: `"A"`, `"B"`, `"C"` of `"D"`
- `antwoord_orientatie` — meerkeuzevraag 2: `"A"`, `"B"`, `"C"` of `"D"`

Gebruik voor het speelgoed-model `AlternatingLeastSquares(factors=50, random_state=0)` en train het met `als_toy_model.fit(sparse_user_item_matrix)`.


## Oefening 1 — Speelgoed-data: pivot-tabel en sparse-matrix

We vertrekken van een kleine dataset met ratings (`user_id`, `song_id`, `rating`). Zet die om naar een **user-item-matrix** met `df.pivot_table(...)` (rijen = users, kolommen = songs, ontbrekende ratings = 0) en daarna naar een **sparse CSR-matrix** met `scipy.sparse.csr_matrix`. Ten slotte train je een `AlternatingLeastSquares`-model op die sparse matrix.


In [19]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

data = {
    'user_id': [1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6],
    'song_id': [1, 2, 1, 3, 4, 5, 5, 3, 5, 2, 4, 6],
    'rating': [4, 3, 5, 2, 4, 1, 3, 5, 2, 4, 3, 5]
}

df = pd.DataFrame(data)


In [20]:
# TODO: zet df om in een user-item-matrix met behulp van een pivot-tabel.
# Rijen = user_id, kolommen = song_id, values = rating, ontbrekende waarden = 0.
user_item_matrix = df.pivot_table(index='user_id', columns='song_id', values='rating', fill_value=0)  # vervang door df.pivot_table(...)

user_item_matrix.head()

song_id,1,2,3,4,5,6
user_id,,,,,,
1,4.0,3.0,0.0,0.0,0.0,0.0
2,5.0,0.0,2.0,0.0,0.0,0.0
3,0.0,0.0,0.0,4.0,2.0,0.0
4,0.0,0.0,5.0,0.0,2.0,0.0
5,0.0,4.0,0.0,3.0,0.0,0.0


In [21]:
# TODO: zet user_item_matrix om naar een sparse matrix in CSR-formaat.
sparse_user_item_matrix = csr_matrix(user_item_matrix.values)  # vervang door csr_matrix(user_item_matrix.values)

sparse_user_item_matrix._shape

(6, 6)

In [23]:
# TODO: maak een AlternatingLeastSquares-model met factors=50 en random_state=0,
# train het op sparse_user_item_matrix en bewaar het model in als_toy_model.
als_toy_model = AlternatingLeastSquares(factors=50, random_state=0)  # vervang door een getraind model
als_toy_model.fit(sparse_user_item_matrix)

100%|██████████| 15/15 [00:00<00:00, 52.11it/s]


In [24]:
# Optioneel (NIET gecheckt): bekijk de aanbevelingen voor user 1.
ids_toy, scores_toy = als_toy_model.recommend(1, sparse_user_item_matrix[1], N=3)
print(ids_toy, scores_toy)


[1 4 5] [ 3.6003431e-03  2.1806993e-03 -5.9697777e-06]


## Oefening 2 — last.fm 360K (optionele uitbreiding, NIET gecheckt)

Deze uitbreiding wordt **niet** automatisch gecheckt. Je kan haar doen voor verdieping, precies zoals in `recommender_systems_oefeningen.ipynb`:

- Download de data met `from implicit.datasets.lastfm import get_lastfm`.
- Weeg de `(item, user)`-matrix met `bm25_weight` uit `implicit.nearest_neighbours` en transponeer daarna naar een `(user, item)` CSR-matrix `user_plays`.
- Train `AlternatingLeastSquares(factors=64, regularization=0.05, alpha=2.0)` en maak aanbevelingen met `.recommend`.

Deze uitbreiding vereist dat `implicit` (en `h5py`) geïnstalleerd zijn via `uv sync`.


In [53]:
from implicit.datasets.lastfm import get_lastfm

artists, users, plays = get_lastfm()
plays

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 17535606 stored elements and shape (292385, 358868)>

In [57]:
from implicit.nearest_neighbours import bm25_weight
weighted = bm25_weight(plays)

user_plays = weighted.tocsc().T

In [58]:
user_plays[:5]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 245 stored elements and shape (5, 292385)>

In [61]:
als_model = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=2.0)
als_model.fit(user_plays)

100%|██████████| 15/15 [00:54<00:00,  3.65s/it]


In [91]:
uid = 10000

recs_user1 = als_model.recommend(userid=uid, user_items=user_plays[uid])
artists[recs_user1[0]]

array(['fall out boy', 'avril lavigne', 'panic at the disco',
       'plain white ts', 'panic! at the disco', '+44', 'katy perry',
       'blink-182', "plain white t's", 'secondhand serenade'],
      dtype=object)

In [96]:
g = 0

while "а" not in "".join(artists[recs_user1[0]]):
    recs_user1 = als_model.recommend(userid=g, user_items=user_plays[g])
    g += 1

artists[recs_user1[0]]

array(['dolphin', 'cheese people', 'punk tv', 'noize mc', 'sakura',
       'Мои Ракеты Вверх', 'animal Джаz', '7раса', 'sunsay',
       'Ляпис Трубецкой'], dtype=object)

## Meerkeuzevragen

### Vraag 1 — Impliciete vs. expliciete feedback
In een recommender-systeem met **impliciete feedback** tellen we hoe vaak een gebruiker met een item interageert (bv. hoe vaak iemand een nummer afspeelt). Welke van de volgende is een **voorbeeld van expliciete feedback**?

- **A)** Het aantal keren dat een gebruiker een nummer heeft afgespeeld.
- **B)** Een gebruiker geeft een rating van 5 sterren aan een film.
- **C)** Het aantal keren dat een gebruiker een pagina heeft bekeken.
- **D)** Een gebruiker klikt op een aanbevolen (onbekend) nummer en luistert het volledig af.

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_feedback`.

### Vraag 2 — Oriëntatie van de user-item-matrix
De `implicit`-bibliotheek verwacht voor ALS dat de ingevoerde matrix **users als rijen** en **items als kolommen** heeft. De `get_lastfm()`-functie geeft echter een matrix terug waarvan elke rij een artiest is en elke kolom een gebruiker. Wat moet je doen om dit te corrigeren?

- **A)** Niets; de matrix werkt al correct.
- **B)** De matrix transponeren met `.T` en omzetten naar CSR-formaat.
- **C)** De matrix normaliseren met `StandardScaler`.
- **D)** Het model trainen op de originele matrix (rijen = artiesten).

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_orientatie`.


In [ ]:
# Vraag 1: antwoord_feedback = "A", "B", "C" of "D"
antwoord_feedback = "A"

# Vraag 2: antwoord_orientatie = "A", "B", "C" of "D"
antwoord_orientatie = ""
